# 🩺 糖尿病患病風險評估 API 教學：實作 `/train` 線上重訓與模型熱載入端點

> 對象：修習邏輯迴歸分類器 / 模型部署課程的學生
> 對應程式碼：`app.py` 第 88~106 行的 `train_endpoint` 與 `train_save.py` 的 `train_and_save_model` 函數
>
> 這份 Notebook 會帶你**一步步拆解**線上模型重新訓練、超參數微調 (C 值) 與模型熱重載 (Hot Reload) 機制，並使用 `TestClient` 模擬完整 API 呼叫流程。

## 🎯 學習目標

- [ ] 理解「線上重訓 (Online Retraining)」與「模型熱重載」的概念
- [ ] 掌握邏輯迴歸中正則化強度逆數 $C$ 與測試集切分比例 `test_size` 對模型擬合的影響
- [ ] 看懂 FastAPI 訓練端點的請求模型 (`TrainRequest`) 與回傳格式
- [ ] 理解 `train_and_save_model()` 如何將模型與 `StandardScaler` 一同序列化儲存至 `.joblib`
- [ ] 能使用 `TestClient` 發送 POST `/train` 請求，並驗證線上重訓後的訓練與測試準確率

## 📌 背景：`/train` 端點運作架構

在智慧醫療應用中，我們可能需要微調邏輯迴歸模型的正則化參數 $C$（控制對係數過大的懲罰程度），或調整訓練與測試集切分。
我們希望**不用關閉或重啟伺服器**就能完成模型更新：

```
[收到 TrainRequest 請求: C, test_size]
       │
       ▼
1. train_and_save_model(...)  ──> 重新訓練邏輯迴歸與標準化器，覆蓋 diabetes_model.joblib
       │
       ▼
2. 更新伺服器全域記憶體變數    ──> 更新全域 model, scaler, odds_ratios
       │
       ▼
3. 回傳結構化 JSON 回應       ──> 回傳訓練/測試準確率與時間戳記
```

In [ ]:
# ============================================
# 0. 載入套件與環境設定
# ============================================
import os, sys
from pprint import pprint

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from train_save import train_and_save_model
print("✅ 成功匯入 train_and_save_model 模組！")

---
## Part 1：測試模型訓練與序列化儲存 (`train_and_save_model`)

先直接呼叫 `train_and_save_model` 觀察訓練流程、特徵標準化與計算出的勝算比 (Odds Ratio)。

In [ ]:
result = train_and_save_model(
    test_size=0.2,
    C=1.0,
    random_state=42
)

print("\n--- 訓練回傳元數據摘要 ---")
print(f"訓練集準確率: {result['train_accuracy']:.4f}")
print(f"測試集準確率: {result['test_accuracy']:.4f}")
print("臨床危險因子勝算比 (Odds Ratios):")
pprint(result['odds_ratios'])

---
## Part 2：定義訓練請求的 Pydantic 模型

在 FastAPI 中，我們定義 `TrainRequest` 來進行輸入驗證：

In [ ]:
from pydantic import BaseModel, Field

class TrainRequest(BaseModel):
    C: float = Field(default=1.0, gt=0, description="正則化強度逆數 C (必須大於 0)")
    test_size: float = Field(default=0.2, ge=0.1, le=0.5, description="測試集切分比例 (0.1 ~ 0.5)")

# 測試建立一個請求物件
sample_req = TrainRequest(C=0.5, test_size=0.25)
print("請求模型實例化成功：", sample_req.model_dump())

---
## Part 3：建立 FastAPI 應用與 `/train` 線上重訓端點

建立 FastAPI App，並實作 `/train` 路由處理器。

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI(title="Diabetes Training API")

MODEL_PATH = "diabetes_model.joblib"
GLOBAL_STATE = {}

@app.post("/train", summary="線上重訓練模型")
def train_endpoint(req: TrainRequest):
    try:
        new_meta = train_and_save_model(
            test_size=req.test_size,
            C=req.C,
            model_path=MODEL_PATH
        )
        GLOBAL_STATE["model"] = new_meta["model"]
        GLOBAL_STATE["scaler"] = new_meta["scaler"]
        GLOBAL_STATE["odds_ratios"] = new_meta["odds_ratios"]
        
        return {
            "status": "success",
            "message": "模型線上重新訓練完成並已熱載入！",
            "train_accuracy": new_meta["train_accuracy"],
            "test_accuracy": new_meta["test_accuracy"],
            "timestamp": new_meta["timestamp"]
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"訓練失敗: {str(e)}")

print("✅ FastAPI /train 端點註冊成功！")

---
## Part 4：使用 `TestClient` 模擬呼叫 `/train` 端點

透過 FastAPI 內建的 `TestClient` 發送 POST 請求驗證。

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# 發送測試請求 (設定 C=2.0, test_size=0.3)
payload = {
    "C": 2.0,
    "test_size": 0.3
}

response = client.post("/train", json=payload)
print("HTTP 狀態碼：", response.status_code)
print("API 回傳內容：")
pprint(response.json())